# PI3Kα (4JPS) — Step 7: Prepare the 10 shortlisted tryptanthrin analogs

**Project:** Tryptanthrin anticancer drug discovery · **Member:** Ayana · **Target:** PI3Kα / `PI3Kalpha`

**Input:** `master_docking_handover.csv` (from your completed ML work). Only the 10 rows with `Target == PI3Kalpha` are used. We preserve `Analog_ID` and metadata.

**Output:** individual RDKit 3D SDF and Meeko ligand PDBQT files, a selected-shortlist CSV, and preparation/quality-control report. **This notebook does not dock the analogs.**

**Validated prior milestone:** Step 6 recovered alpelisib 1LT in 4JPS at top-pose RMSD ≈ 0.604 Å; use the exact Step 6 receptor/grid/settings in the *subsequent* docking notebook. Retain documented receptor-exclusion caveats.

Run cells in order in **Google Colab**. Do not rerun Step 5 or Step 6 here.


## 1. Install and verify dependencies

In [1]:
%pip -q install rdkit meeko gemmi pandas numpy
import sys, subprocess
from pathlib import Path
from rdkit import Chem, rdBase
from rdkit.Chem import AllChem
import pandas as pd
import numpy as np
print('Python:',sys.version.split()[0], 'RDKit:',rdBase.rdkitVersion)
check=subprocess.run([sys.executable,'-m','meeko.cli.mk_prepare_ligand','--help'],capture_output=True,text=True)
print('Meeko ligand CLI:', 'OK' if check.returncode==0 else 'ERROR')
if check.returncode: raise RuntimeError(check.stderr[-2000:])

Python: 3.13.18 RDKit: 2026.09.2
Meeko ligand CLI: OK


## 2. Upload the master shortlist CSV (not a ZIP)

In [3]:
from google.colab import files
uploaded=files.upload()
expected='master_docking_handover.csv'
if expected not in uploaded:
    candidates=[k for k in uploaded if k.lower().endswith('.csv')]
    if len(candidates)!=1: raise RuntimeError('Upload one master_docking_handover.csv file.')
    filename=candidates[0]
else: filename=expected
source=pd.read_csv(filename,keep_default_na=True)
needed={'Target','Analog_ID','SMILES','Selection_rank'}
assert needed.issubset(source.columns), f'Missing columns: {needed-set(source.columns)}'
sel=source[source['Target'].astype(str).str.strip().eq('PI3Kalpha')].copy()
sel=sel.sort_values('Selection_rank')
assert len(sel)==10, f'Expected 10 PI3Kalpha candidates, got {len(sel)}'
assert sel['Analog_ID'].notna().all() and sel['SMILES'].notna().all()
assert sel['Analog_ID'].is_unique, 'Duplicate Analog_ID within the PI3Kalpha shortlist'
assert all(str(x).startswith('TRY_') for x in sel.Analog_ID)
root=Path('PI3Kalpha_Step7_Ligand_Preparation')
ligands=root/'ligands'; logs=root/'logs'
for p in [root,ligands,logs]:p.mkdir(parents=True,exist_ok=True)
sel.to_csv(root/'PI3Kalpha_selected_top10.csv',index=False)
print('Selected PI3Kalpha analogs:',len(sel))
display(sel[['Selection_rank','Analog_ID','SMILES','Consensus_pIC50','QED','SA_score']])

Saving master_docking_handover.csv to master_docking_handover.csv
Selected PI3Kalpha analogs: 10


,Selection_rank,Analog_ID,SMILES,Consensus_pIC50,QED,SA_score
20,1,TRY_c2155e898021da1c,O=C1c2cc(O)ccc2-n2c1nc1c(N3CCOCC3)cccc1c2=O,6.44444,0.560199,2.508959
21,2,TRY_c6b27d68ce6ec718,CC(=O)Nc1ccc2c(c1)-n1c(nc3c(N4CCOCC4)cccc3c1=O...,6.36582,0.560274,2.506605
22,3,TRY_40b0c0e6de99494a,COc1ccc2c(c1)C(=O)c1nc3c(N4CCOCC4)cccc3c(=O)n1-2,6.33990,0.540488,2.422261
23,4,TRY_f751442f427b1d4d,O=C1c2c(N3CCOCC3)cccc2-n2c1nc1c(O)cccc1c2=O,6.30231,0.560199,2.587839
24,5,TRY_6db5e7a7802259ab,O=C1c2cc(C3CC3)ccc2-n2c1nc1cc(N3CCOCC3)ccc1c2=O,6.29428,0.540506,2.488290
25,6,TRY_6bf367653fed7969,O=C1c2c(O)cccc2-n2c1nc1cccc(N3CCOCC3)c1c2=O,6.26524,0.560199,2.569880
26,7,TRY_c88335488290afb4,NC(=O)c1ccc2c(c1)C(=O)c1nc3cc(N4CCOCC4)ccc3c(=...,6.25409,0.556121,2.444439
27,8,TRY_e514ff1b3f600530,COc1ccc2nc3n(c(=O)c2c1)-c1c(cccc1N1CCOCC1)C3=O,6.22360,0.540488,2.458244
28,9,TRY_bafc3535a3d34b71,COc1cccc2c(=O)n3c(nc12)C(=O)c1cc(N2CCOCC2)ccc1-3,6.22232,0.540488,2.411619
29,10,TRY_783966fbcbaa4f5c,CC(C)c1c(N2CCOCC2)ccc2c1-n1c(nc3ccccc3c1=O)C2=O,6.21648,0.539044,2.577449


## 3. Generate explicit-H 3D conformers with RDKit and optimise with MMFF94

The input SMILES are the source of chemistry. Every compound is checked for parsing, stereochemistry and geometry. We do **not** silently fix invalid structures. One reproducible seed is used per ligand; several conformers are tried and the lowest MMFF energy successful conformer is retained. This is a **preparation protocol**, not experimental validation of stereochemistry, tautomer, or protonation.

In [4]:
import json
from rdkit.Chem import rdMolDescriptors

prep_records=[]
for _,row in sel.iterrows():
    aid=str(row['Analog_ID']); smi=str(row['SMILES'])
    rec={'Analog_ID':aid,'Selection_rank':int(row['Selection_rank']), 'SMILES':smi,
         'sdf_ok':False,'pdbqt_ok':False,'status':'PENDING','error':'',
         'mmff_energy_kcal_mol':np.nan,'num_conformers_generated':0,
         'unassigned_stereocenters':None,'heavy_atom_count':None}
    try:
        mol=Chem.MolFromSmiles(smi)
        if mol is None: raise ValueError('RDKit cannot parse SMILES')
        Chem.SanitizeMol(mol)
        rec['heavy_atom_count']=mol.GetNumHeavyAtoms()
        chiral=Chem.FindMolChiralCenters(mol,includeUnassigned=True)
        rec['unassigned_stereocenters']=sum(label=='?' for _,label in chiral)
        if rec['unassigned_stereocenters']:
            raise ValueError(f"{rec['unassigned_stereocenters']} unspecified stereocenter(s); resolve before docking")
        mh=Chem.AddHs(mol)
        params=AllChem.ETKDGv3()
        params.randomSeed=42+int(row['Selection_rank'])
        params.numThreads=0
        ids=list(AllChem.EmbedMultipleConfs(mh,numConfs=8,params=params))
        rec['num_conformers_generated']=len(ids)
        if not ids: raise ValueError('ETKDG failed for every conformer')
        if not AllChem.MMFFHasAllMoleculeParams(mh):
            raise ValueError('MMFF94 parameters not available for this ligand')
        opt=AllChem.MMFFOptimizeMoleculeConfs(mh,numThreads=0,maxIters=500,mmffVariant='MMFF94')
        success=[(float(energy),conf_id) for conf_id,(code,energy) in zip(ids,opt) if code==0]
        if not success: raise ValueError('No MMFF94 conformer converged')
        energy,best_id=min(success)
        chosen=Chem.Mol(mh)
        for cid in [c.GetId() for c in chosen.GetConformers() if c.GetId()!=best_id]: chosen.RemoveConformer(cid)
        chosen.GetConformer().SetId(0)
        chosen.SetProp('_Name',aid);chosen.SetProp('Analog_ID',aid)
        chosen.SetProp('Original_SMILES',smi)
        chosen.SetProp('Target','PI3Kalpha')
        sdf=ligands/f'{aid}.sdf'
        w=Chem.SDWriter(str(sdf));w.write(chosen);w.close()
        reloaded=next((x for x in Chem.SDMolSupplier(str(sdf),removeHs=False) if x is not None),None)
        if reloaded is None or reloaded.GetNumHeavyAtoms()!=mol.GetNumHeavyAtoms():
            raise RuntimeError('Written SDF failed molecule count/chemistry reload')
        rec['mmff_energy_kcal_mol']=energy
        rec['sdf_ok']=True
        rec['status']='SDF_READY'
    except Exception as e:
        rec['status']='SDF_FAILED';rec['error']=str(e)
    prep_records.append(rec)
prep=pd.DataFrame(prep_records)
display(prep[['Analog_ID','status','num_conformers_generated','mmff_energy_kcal_mol','error']])
print('SDF succeeded:',int(prep.sdf_ok.sum()),'of',len(prep))

,Analog_ID,status,num_conformers_generated,mmff_energy_kcal_mol,error
0,TRY_c2155e898021da1c,SDF_READY,8,140.658450,
1,TRY_c6b27d68ce6ec718,SDF_READY,8,136.908536,
2,TRY_40b0c0e6de99494a,SDF_READY,8,154.275346,
3,TRY_f751442f427b1d4d,SDF_READY,8,147.447063,
4,TRY_6db5e7a7802259ab,SDF_READY,8,139.078140,
5,TRY_6bf367653fed7969,SDF_READY,8,160.362908,
6,TRY_c88335488290afb4,SDF_READY,8,113.920236,
7,TRY_e514ff1b3f600530,SDF_READY,8,149.547968,
8,TRY_bafc3535a3d34b71,SDF_READY,8,149.125826,
9,TRY_783966fbcbaa4f5c,SDF_READY,8,156.467596,


SDF succeeded: 10 of 10


## 4. Convert each successful ligand SDF into PDBQT with Meeko

Capture the exact command and log for every molecule. Failure does not silently skip a candidate in the final report.

In [5]:
for rec in prep_records:
    aid=rec['Analog_ID']
    if not rec['sdf_ok']:continue
    sdf=ligands/f'{aid}.sdf'; pdbqt=ligands/f'{aid}.pdbqt'
    cmd=[sys.executable,'-m','meeko.cli.mk_prepare_ligand','-i',str(sdf),'-o',str(pdbqt)]
    p=subprocess.run(cmd,capture_output=True,text=True)
    log='COMMAND: '+' '.join(cmd)+'\nRETURN CODE: '+str(p.returncode)+'\n'+p.stdout+'\n'+p.stderr
    (logs/f'{aid}_meeko.log').write_text(log,encoding='utf-8')
    good=p.returncode==0 and pdbqt.exists() and pdbqt.stat().st_size>0
    rec['pdbqt_ok']=good
    if good:
        rec['status']='READY_FOR_DOCKING'
    else:
        rec['status']='PDBQT_FAILED'
        rec['error']=(p.stderr or p.stdout)[-1200:]
prep=pd.DataFrame(prep_records)
prep.to_csv(root/'PI3Kalpha_ligand_preparation_status.csv',index=False)
display(prep[['Analog_ID','sdf_ok','pdbqt_ok','status','error']])
print('PDBQT ready:',int(prep.pdbqt_ok.sum()),'of',len(prep))
if not prep.pdbqt_ok.all():
    print('ATTENTION: At least one ligand failed. Inspect logs and resolve before claiming all 10 prepared.')

,Analog_ID,sdf_ok,pdbqt_ok,status,error
0,TRY_c2155e898021da1c,True,True,READY_FOR_DOCKING,
1,TRY_c6b27d68ce6ec718,True,True,READY_FOR_DOCKING,
2,TRY_40b0c0e6de99494a,True,True,READY_FOR_DOCKING,
3,TRY_f751442f427b1d4d,True,True,READY_FOR_DOCKING,
4,TRY_6db5e7a7802259ab,True,True,READY_FOR_DOCKING,
5,TRY_6bf367653fed7969,True,True,READY_FOR_DOCKING,
6,TRY_c88335488290afb4,True,True,READY_FOR_DOCKING,
7,TRY_e514ff1b3f600530,True,True,READY_FOR_DOCKING,
8,TRY_bafc3535a3d34b71,True,True,READY_FOR_DOCKING,
9,TRY_783966fbcbaa4f5c,True,True,READY_FOR_DOCKING,


PDBQT ready: 10 of 10


## 5. Save quality-control metadata and download Step 7 package

In [6]:
import shutil,hashlib
report={
 'target':'PI3Kalpha','pdb_id':'4JPS',
 'source_shortlist_filename':Path(filename).name,
 'source_shortlist_sha256':hashlib.sha256(Path(filename).read_bytes()).hexdigest(),
 'selected_analog_count':int(len(sel)),
 'sdf_success_count':int(prep.sdf_ok.sum()),
 'pdbqt_success_count':int(prep.pdbqt_ok.sum()),
 'all_10_ready':bool(prep.pdbqt_ok.all()),
 'method':'RDKit ETKDGv3 (8 conformers, deterministic seeds) + MMFF94; Meeko PDBQT preparation',
 'caveats':['Ligand protonation and tautomer states require scientific review',
            'A successful conversion does not validate ligand binding or docking',
            'Retain Analog_ID when merging with model outputs'],
 'next':'Dock ready compounds using exactly the receptor/grid/settings validated by Step 6'
}
(root/'PI3Kalpha_step7_report.json').write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
archive=shutil.make_archive(str(root),'zip',str(root))
print('Output:',archive,'Size:',Path(archive).stat().st_size,'bytes')
print('Prepared:',report['pdbqt_success_count'],'of 10; all ready:',report['all_10_ready'])
files.download(archive)

Output: /content/PI3Kalpha_Step7_Ligand_Preparation.zip Size: 26755 bytes
Prepared: 10 of 10; all ready: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## After Step 7

Keep the **executed `.ipynb`** and output ZIP. If `all_10_ready` is false, resolve failures before Step 8. Step 8 will use the *same validated Step 6 receptor and box*, not a new receptor. Final GitHub layout follows your teammate's structure: `06_Molecular_Docking/PI3Kalpha/{notebooks,structures,ligands,results,analysis}`. Don't upload intermediate debugging notebooks to the final deliverable.
